In [1]:
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics import silhouette_score
from sklearn.cluster import KMeans

categories = ['sci.space', 'rec.autos', 'talk.politics.misc']
print(f'Loading 20 newsgroups dataset for categories: {categories}')
data = fetch_20newsgroups(subset='all', categories=categories, shuffle=True, random_state=42)

# TF-IDF Vectorization
vectorizer = TfidfVectorizer(stop_words='english', max_features=10000)
X_tfidf = vectorizer.fit_transform(data.data)
print(f'TF-IDF Matrix shape: {X_tfidf.shape}')

Loading 20 newsgroups dataset for categories: ['sci.space', 'rec.autos', 'talk.politics.misc']
TF-IDF Matrix shape: (2752, 10000)


In [2]:
component_counts = [50, 100, 200]
results = []
topic_logs = []

for n in component_counts:
    # TruncatedSVD (LSA)
    svd = TruncatedSVD(n_components=n, random_state=42)
    X_reduced = svd.fit_transform(X_tfidf)

    explained_variance = svd.explained_variance_ratio_.sum()

    # Clustering (KMeans)
    kmeans = KMeans(n_clusters=3, random_state=42, n_init='auto')
    cluster_labels = kmeans.fit_predict(X_reduced)
    s_score = silhouette_score(X_reduced, cluster_labels)

    results.append({'components': n, 'explained_variance': explained_variance, 'silhouette_score': s_score})

    terms = vectorizer.get_feature_names_out()
    topic_logs.append(f'\n--- Components: {n} ---')
    for i, comp in enumerate(svd.components_[:5]):
        top_terms = [terms[idx] for idx in comp.argsort()[-10:][::-1]]
        topic_logs.append(f'Topic {i+1}: {", ".join(top_terms)}')

results_df = pd.DataFrame(results)
results_df.to_csv('svd_results.csv', index=False)
with open('lsa_topic_terms.txt', 'w') as f:
    f.write('\n'.join(topic_logs))

display(results_df)

,components,explained_variance,silhouette_score
0,50,0.171077,0.014622
1,100,0.259946,0.005685
2,200,0.385563,0.033791
